In [ ]:
# ── PARAMETERS CELL ──────────────────────────────────────────────────────────
# The 'parameters' tag is already set in this cell's metadata.
# Fabric reads that tag and overrides these defaults with real pipeline values.
# Pipeline wiring (already confirmed in your screenshot):
#   p_pipeline_run_id → @pipeline().RunId
#   p_file_name       → @item().name
#   p_files_written   → @string(activity('Copy_SFTP_to_Bronze').output.filesWritten)
#   p_bronze_path     → @concat('Files/bronze/sales/', item().name)
# ─────────────────────────────────────────────────────────────────────────────
p_pipeline_run_id = "manual_run"
p_file_name       = "unknown_file.csv"
p_bronze_path     = "Files/bronze/sales/"
p_FileModifiedTime = "1900-01-01T00:00:000"

In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime, timezone
# ── Write audit row ───────────────────────────────────────────────────────────
AUDIT_TABLE = "bronze_audit_log"

now_utc  = datetime.now(timezone.utc)
audit_id = int(now_utc.timestamp() * 1000)
now_str  = now_utc.strftime("%Y-%m-%d %H:%M:%S")


spark.sql(f"""
    INSERT INTO {AUDIT_TABLE}
        (AuditId, PipelineRunId, FileName, FileModifiedTime,
         DataLoadedDate, BronzePath, Status, ErrorMessage)
    VALUES (
        {audit_id},
        '{p_pipeline_run_id}',
        '{p_file_name}',
        '{p_FileModifiedTime}',
        CAST('{now_str}' AS TIMESTAMP),
        '{p_bronze_path}',
        'Success',
        NULL
    )
""")

